# Resolución de Entidades: Desambiguación de Afiliaciones

## Planteamiento del Problema

Para el caso de las `afiliaciones`, no hay duplicados exactos como tal sino que varias de ellas están escritas de forma diferente, es decir, hay representaciones alternativas de una misma afiliación.

In [44]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import unicodedata
from ast import literal_eval
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

In [45]:
DATA_DIR = Path("data/entities")
OUTPUT_DIR = Path("data/extra")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

affiliations_path = DATA_DIR / "afiliaciones_ecuador.csv"
if not affiliations_path.exists():
    affiliations_path = Path("afiliaciones_ecuador.csv")

df_aff = pd.read_csv(affiliations_path)

if "affilname_es" not in df_aff.columns:
    if "affilname" not in df_aff.columns:
        raise KeyError("Expected either 'affilname_es' or 'affilname' in the affiliations file.")
    df_aff["affilname_es"] = df_aff["affilname"]

print(f"Archivo de afiliaciones: {affiliations_path}")
print(f"Forma de df_aff: {df_aff.shape}")
print(f"Total de afiliaciones unicas: {df_aff['afid'].nunique()}")
df_aff.head()

Archivo de afiliaciones: data\entities\afiliaciones_ecuador.csv
Forma de df_aff: (8143, 5)
Total de afiliaciones unicas: 8143


,afid,affilname,affiliation-city,affiliation-country,affilname_es
0,60278953,Universidad Bolivariana del Ecuador,Duran,Ecuador,Universidad Bolivariana del Ecuador
1,133242834,Instituto de Investigación Multidisciplinaria ...,NaN,Ecuador,Instituto de Investigación Multidisciplinaria ...
2,60108912,Universidad Técnica de Manabí,Portoviejo,Ecuador,Universidad Técnica de Manabí
3,131940052,Unversidad Estatal de Milagro,Milagro,Ecuador,Unversidad Estatal de Milagro
4,60072064,Universidad Técnica Particular de Loja,Loja,Ecuador,Universidad Técnica Particular de Loja


In [46]:
df_aff.isnull().sum()

afid                      0
affilname                 0
affiliation-city       2355
affiliation-country       0
affilname_es              0
dtype: int64

Un ejemplo claro es el de la **Escuela Politécnica Nacional**:

In [47]:
similar_epn = [
    60072054, 101306727, 115317659,
    127982025, 128310067, 131445344,
    121697195, 132291228, 131334134,
    122120542, 123346277, 123988974,
    123789075, 124101307, 130722865,
    130722929, 130838989, 130472131,
]

In [48]:
df_aff[df_aff["afid"].isin(similar_epn)]

,afid,affilname,affiliation-city,affiliation-country,affilname_es
71,60072054,Escuela Politécnica Nacional,Quito,Ecuador,Escuela Politécnica Nacional
1743,131445344,Escuela Politécnic a Nacional,Quito,Ecuador,Escuela Politécnic a Nacional
1766,115317659,Escuela Poliécnica Nacional,Quito,Ecuador,Escuela Poliécnica Nacional
2308,132291228,Escuela Politécnica Nacional (National Polytec...,NaN,Ecuador,Escuela Politécnica Nacional (National Polytec...
2340,123346277,Escuela Poltécnica Nacional,Quito,Ecuador,Escuela Poltécnica Nacional
3119,130472131,Politechnical National School,Toledo,Ecuador,Politechnical National School
3190,121697195,Escuela Politécnica,NaN,Ecuador,Escuela Politécnica
3547,128310067,Escuela PolitCrossed D Sign©cnica Nacional,Quito,Ecuador,Escuela PolitCrossed D Sign©cnica Nacional
3590,127982025,Escuela Politcnica,NaN,Ecuador,Escuela Politcnica
3993,131334134,Escuela Politéctnica Nacional,Quito,Ecuador,Escuela Politéctnica Nacional


In [49]:
df_aff[df_aff["afid"].isin(similar_epn)].to_csv(
    OUTPUT_DIR / "afiliaciones_similares_epn.csv",
    index=False,
)

Otros ejemplos también son:

In [50]:
df_aff[
    df_aff["afid"].isin(
        [132217264, 132462163, 126803765, 133300960, 100358737, 132157804]
    )
]

,afid,affilname,affiliation-city,affiliation-country,affilname_es
988,132462163,Academia de Guerra del Ejército,Sangolquí,Ecuador,Academia de Guerra del Ejército
2335,132217264,Academia de Guerra del Ejercito,Quito,Ecuador,Academia de Guerra del Ejercito
4311,126803765,Academia de Defensa Militar,Conjunta,Ecuador,Academia de Defensa Militar
5126,133300960,Academia de Defensa Militar Conjunta,Sangolquí,Ecuador,Academia de Defensa Militar Conjunta
7798,100358737,ACUATECNOS,Guayaquil,Ecuador,ACUATECNOS
7933,132157804,Acuatecnos,Guayaguil,Ecuador,Acuatecnos


Esto puede deberse a la forma en que Scopus maneja las relaciones de las afiliaciones, ya que el nombre y la ciudad puede no variar, sin embargo, si puede tratarse de una sucursal de esa afiliación, por lo cual se puede definir los siguientes errores:
- **Variaciones de idioma**
- **Errores tipográficos**
- **Uso de acrónimos**
- **Inconsistencia en ubicación geográfica**

## Exploración de Datos

### Longitud de nombres

In [51]:
df_aff["name_length"] = df_aff["affilname_es"].str.len()
print(df_aff["name_length"].describe())
df_aff = df_aff.drop(columns=["name_length"])

count    8143.000000
mean       35.811249
std        20.452325
min         2.000000
25%        22.000000
50%        32.000000
75%        46.000000
max       218.000000
Name: name_length, dtype: float64


### Caracteres especiales

In [52]:
special_chars = df_aff[df_aff["affilname_es"].str.contains(r"[^A-Za-zÁÉÍÓÚáéíóúÑñÜü0-9 ]", na=False)]
print(f"Afiliaciones con caracteres especiales (sin contar acentos): {len(special_chars)}")
special_chars

Afiliaciones con caracteres especiales (sin contar acentos): 2266


,afid,affilname,affiliation-city,affiliation-country,affilname_es
21,133088229,Centro de Investigación y Desarrollo en Nanote...,Guayaquil,Ecuador,Centro de Investigación y Desarrollo en Nanote...
37,60104441,Universidad de las Americas - Ecuador,Quito,Ecuador,Universidad de las Americas - Ecuador
41,60113878,"Universidad Nacional de Educación, Ecuador",Azogues,Ecuador,"Universidad Nacional de Educación, Ecuador"
45,133323533,Ministerio de Ecucación del Ecuador (MINEDUC),Ambato,Ecuador,Ministerio de Ecucación del Ecuador (MINEDUC)
50,128862238,Universidad Autónoma de los Andes (UNIANDES),Quevedo,Ecuador,Universidad Autónoma de los Andes (UNIANDES)
...,...,...,...,...,...
8122,117679382,Faculté D'agronomie Et De Médicine Vétérinaire...,Quito,Ecuador,Faculté D'agronomie Et De Médicine Vétérinaire...
8125,117678904,Université Centrale de L'equateur,Quito,Ecuador,Université Centrale de L'equateur
8128,101357107,Instituto Nacional de Higiene 'Leopoldo Izquie...,NaN,Ecuador,Instituto Nacional de Higiene 'Leopoldo Izquie...
8129,114594823,Anglo-Ecuadorian Oilfields Ltd,NaN,Ecuador,Anglo-Ecuadorian Oilfields Ltd


In [53]:
with_numbers = df_aff[df_aff["affilname_es"].str.contains(r"[0-9]", na=False)]
with pd.option_context("display.max_rows", None):
    print(f"Afiliaciones con números: {len(with_numbers)}")
    with_numbers = with_numbers.sort_values("affilname_es")
    display(with_numbers)

Afiliaciones con números: 181


,afid,affilname,affiliation-city,affiliation-country,affilname_es
5834,126155917,13D03 Jipijapa-Puerto López. Ministerio de Sal...,Jipijapa,Ecuador,13D03 Jipijapa-Puerto López. Ministerio de Sal...
7726,115652939,16-310 Quito,NaN,Ecuador,16-310 Quito
5490,122819394,360Life Technologies,Quito,Ecuador,360Life Technologies
5050,124376057,3A Composites Research and Development,Guayaquil,Ecuador,3A Composites Research and Development
107,129115108,3Diversity,"Quito, Pichincha",Ecuador,3Diversity
1042,129674606,3Diversity,Quito,Ecuador,3Diversity
4129,128914072,7CargoCorp,Guayaquil,Ecuador,7CargoCorp
6623,122910175,AInstituto Tecnológico Superior 17 de Julio-Ya...,NaN,Ecuador,AInstituto Tecnológico Superior 17 de Julio-Ya...
4305,126587795,Agregado I. Categoría SENESCYT REG-INV-17-02036,Machala,Ecuador,Agregado I. Categoría SENESCYT REG-INV-17-02036
3876,127025813,AgroG2Ec S.A.,Quito,Ecuador,AgroG2Ec S.A.


La mayoria de afiliaciones que cuentan con numeros en el campo `affilname` en realidad son direcciones fisicas.  
Posibles Affs (con números) validos:  
106760808, 132621188, 114129821, 119933429, 114347231, 119084753, 131585760, 114783230, 114334903, 115847150, 116363379, 127241392, 122910175, 118184979, 123222881, 130054373, 121019933, 117876872, 120560792, 120561484, 120871990, 122161570, 126155917, 124335735, 121544468, 122969618, 123095409, 123342062, 131296976, 122819394, 124520204, 123399984, 130768137, 123658550, 124817166, 124142001, 124376057, 125063323, 124998779, 125108503, 127163551, 127003733, 127283381, 112568774, 127393723, 125686069, 125790044, 126364778, 125643866, 126413410, 126587795, 126245280, 126889915, 126291546, 127357760, 127579876, 127579955, 125781106, 100942116, 128914072, 122928960, 131167099, 129064944, 129217139, 127814983, 127025813, 128190086, 128186625, 126950111,  100729671, 128075467, 127013362, 128615871, 128828308, 129217411, 129488444, 121022423, 129856019, 129821360, 129774506, 126774636, 131200354, 124151922, 130001440, 130383661, 130496643, 116600646, 132228596, 131998230, 132785370, 132608539, 119056721, 131352547, 127756530, 130950602, 131096996, 126508162, 132235420, 132291696, 126571443, 129674606, 132478571, 132441873, 132708510, 132348260, 132348260, 132313451, 133286310, 133239592, 132684858, 133107458, 125108474, 129115108, 132830147, 133293118

### Palabras más frecuentes

In [54]:
all_words = " ".join(df_aff["affilname_es"]).lower().split()
common_words = Counter(all_words).most_common(100)
print(common_words)

[('de', 3469), ('hospital', 814), ('y', 704), ('del', 682), ('of', 635), ('instituto', 613), ('universidad', 507), ('la', 443), ('ecuador', 440), ('centro', 427), ('nacional', 354), ('salud', 342), ('superior', 338), ('and', 333), ('en', 298), ('university', 260), ('investigación', 255), ('general', 227), ('research', 223), ('tecnológico', 221), ('unidad', 209), ('fundación', 207), ('the', 202), ('ministerio', 190), ('médico', 167), ('for', 165), ('institute', 159), ('escuela', 153), ('educativa', 153), ('el', 144), ('para', 141), ('national', 140), ('center', 138), ('pública', 130), ('san', 121), ('ecuatoriana', 120), ('clínica', 116), ('ciencias', 115), ('grupo', 114), ('manabí', 111), ('departamento', 111), ('quito', 108), ('guayaquil', 108), ('facultad', 107), ('investigaciones', 102), ('social', 102), ('s.a.', 102), ('health', 99), ('desarrollo', 98), ('ecuatoriano', 89), ('ecuadorian', 88), ('foundation', 87), ('especialidades', 84), ('e', 79), ('iess', 79), ('politécnica', 78), 

Posibles categorías identificadas: hospital, instituto, universidad, centro, fundación, ministerio, escuela, clinica, grupo, departamento, facultad, laboratorio, corporación, sociedad, otros.

In [55]:
print(len(all_words))
print(len(set(all_words)))

38662
7648


## Normalizar

In [56]:
def normalize_text(text):
    text = text.lower()
    # Quitar acentos y simbolos raros
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("utf-8")
    # Quitar numeros
    text = re.sub(r"[^a-z\s]", " ", text)
    # Colapsar espacios multiples
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [57]:
df_aff["affil_clean"] = df_aff["affilname_es"].apply(normalize_text)
df_aff[df_aff["afid"].isin(similar_epn)].head()

,afid,affilname,affiliation-city,affiliation-country,affilname_es,affil_clean
71,60072054,Escuela Politécnica Nacional,Quito,Ecuador,Escuela Politécnica Nacional,escuela politecnica nacional
1743,131445344,Escuela Politécnic a Nacional,Quito,Ecuador,Escuela Politécnic a Nacional,escuela politecnic a nacional
1766,115317659,Escuela Poliécnica Nacional,Quito,Ecuador,Escuela Poliécnica Nacional,escuela poliecnica nacional
2308,132291228,Escuela Politécnica Nacional (National Polytec...,NaN,Ecuador,Escuela Politécnica Nacional (National Polytec...,escuela politecnica nacional national polytech...
2340,123346277,Escuela Poltécnica Nacional,Quito,Ecuador,Escuela Poltécnica Nacional,escuela poltecnica nacional


In [58]:
acronimos = (
    df_aff["affilname_es"]
    .str.findall(r"\(([A-Z](?:\.?[A-Z]){1,})\)")
    .explode()
    .dropna()
    .drop_duplicates()
)
acronimos = acronimos.sort_values().reset_index(drop=True)
print(f"Número de acrónimos únicos encontrados: {len(acronimos)}")
acronimos

Número de acrónimos únicos encontrados: 340


0          ABG
1        ABREC
2          ACE
3       AEPPBE
4      AEPROVI
        ...   
335       UTEG
336     UTLVTE
337        UTN
338         WP
339        WWF
Name: affilname_es, Length: 340, dtype: object

In [59]:
with pd.option_context("display.max_rows", None):
    display(df_aff["affiliation-city"].dropna().value_counts())

affiliation-city
Quito                                                       2175
Guayaquil                                                    870
Cuenca                                                       363
Ambato                                                       180
Riobamba                                                     148
Loja                                                         126
Ibarra                                                        65
Santo Domingo                                                 63
Portoviejo                                                    60
Manta                                                         57
Quevedo                                                       51
Esmeraldas                                                    50
Machala                                                       45
Latacunga                                                     41
Santa Elena                                                   31
Manabí  

In [60]:
STOPWORDS = {"de","del","la","el","los","las","y","e","en","para","por","a","al"}

def tokens(text: str):
    toks = [t for t in text.split() if t not in STOPWORDS]
    return " ".join(toks)

## FingerPrint

In [61]:
def fingerprint(text):
    if not text:
        return ""
    
    text = normalize_text(text)
    tokens = text.split(" ")
    tokens = set(tokens)
    tokens = sorted(tokens)
    return " ".join(tokens)

In [62]:
df_aff["fingerprint"] = df_aff["affilname_es"].apply(fingerprint)
df_aff[df_aff["afid"].isin(similar_epn)].head()

,afid,affilname,affiliation-city,affiliation-country,affilname_es,affil_clean,fingerprint
71,60072054,Escuela Politécnica Nacional,Quito,Ecuador,Escuela Politécnica Nacional,escuela politecnica nacional,escuela nacional politecnica
1743,131445344,Escuela Politécnic a Nacional,Quito,Ecuador,Escuela Politécnic a Nacional,escuela politecnic a nacional,a escuela nacional politecnic
1766,115317659,Escuela Poliécnica Nacional,Quito,Ecuador,Escuela Poliécnica Nacional,escuela poliecnica nacional,escuela nacional poliecnica
2308,132291228,Escuela Politécnica Nacional (National Polytec...,NaN,Ecuador,Escuela Politécnica Nacional (National Polytec...,escuela politecnica nacional national polytech...,epn escuela nacional national politecnica poly...
2340,123346277,Escuela Poltécnica Nacional,Quito,Ecuador,Escuela Poltécnica Nacional,escuela poltecnica nacional,escuela nacional poltecnica


In [63]:
fingerprint_groups = (
    df_aff
    .groupby("fingerprint")
    .filter(lambda x: len(x) > 1)
)

fingerprint_groups.sort_values("fingerprint").head(20)

,afid,affilname,affiliation-city,affiliation-country,affilname_es,affil_clean,fingerprint
1732,131412737,Casta Roja Agroindustrial S.A.,Guayaquil,Ecuador,Casta Roja Agroindustrial S.A.,casta roja agroindustrial s a,a agroindustrial casta roja s
5619,122833189,Casta roja agroindustrial S. A,NaN,Ecuador,Casta roja agroindustrial S. A,casta roja agroindustrial s a,a agroindustrial casta roja s
5091,123865589,AndinaGestión S.A.,Quito,Ecuador,AndinaGestión S.A.,andinagestion s a,a andinagestion s
4489,126100122,AndinaGestión S.A.,Quito,Ecuador,AndinaGestión S.A.,andinagestion s a,a andinagestion s
5093,123864947,AndinaGestión S.A,Quito,Ecuador,AndinaGestión S.A,andinagestion s a,a andinagestion s
5967,108598442,Inst. Nac. Autonomo Invest. A.,Quito,Ecuador,Inst. Nac. Autonomo Invest. A.,inst nac autonomo invest a,a autonomo inst invest nac
6936,101152082,Inst. Nac. Autonomo Invest. A.,Guayaquil,Ecuador,Inst. Nac. Autonomo Invest. A.,inst nac autonomo invest a,a autonomo inst invest nac
2733,124315256,Bira Bienes Raíces S.A. (BIRA S.A.),Zaruma,Ecuador,Bira Bienes Raíces S.A. (BIRA S.A.),bira bienes raices s a bira s a,a bienes bira raices s
5215,124095038,BIRA Bienes Raíces S.A.,NaN,Ecuador,BIRA Bienes Raíces S.A.,bira bienes raices s a,a bienes bira raices s
1548,131894774,Manejo y Conservación de Recursos Naturales S....,Guayaquil,Ecuador,Manejo y Conservación de Recursos Naturales S....,manejo y conservacion de recursos naturales s a s,a conservacion de manejo naturales recursos s y


In [64]:
fingerprint_groups[fingerprint_groups["afid"].isin(similar_epn)].head()

,afid,affilname,affiliation-city,affiliation-country,affilname_es,affil_clean,fingerprint
4829,124101307,National Polytechnic School (EPN),Ladrón the Guevara,Ecuador,National Polytechnic School (EPN),national polytechnic school epn,epn national polytechnic school
5249,130722865,National Polytechnic School (EPN),Ladrón the Guevara,Ecuador,National Polytechnic School (EPN),national polytechnic school epn,epn national polytechnic school
5252,130722929,National Polytechnic School (EPN),Ladrón de Guevara,Ecuador,National Polytechnic School (EPN),national polytechnic school epn,epn national polytechnic school


## TF-IDF

In [65]:
unique_names = df_aff["affilname_es"].dropna().unique()
unique_names

array(['Universidad Bolivariana del Ecuador',
       'Instituto de Investigación Multidisciplinaria Perspectivas Globales',
       'Universidad Técnica de Manabí', ...,
       'Quita Normal de Agricultura',
       'South American Development Company', 'Field Director'],
      shape=(7340,), dtype=object)

In [66]:
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3))
tfidf_matrix = vectorizer.fit_transform(unique_names)
cosine_sim = cosine_similarity(tfidf_matrix)

In [67]:
SIMILARITY_THRESHOLD = 0.85
rows, cols = np.where(cosine_sim > SIMILARITY_THRESHOLD)

In [68]:
G = nx.Graph()
G.add_nodes_from(unique_names)

for r, c in zip(rows, cols):
    if r != c:
        G.add_edge(unique_names[r], unique_names[c])

clusters = list(nx.connected_components(G))

In [69]:
name_counts = df_aff["affilname_es"].value_counts().to_dict()
name_map = {}

for cluster in clusters:
    cluster_list = list(cluster)

    # Logic: Pick the name with highest frequency.
    # If tie, pick the longest one (usually more complete).
    standard_name = max(cluster_list, key=lambda x: (name_counts.get(x, 0), len(x)))

    # Map every variation in the cluster to this standard name
    for name in cluster_list:
        name_map[name] = standard_name

In [70]:
df_aff["standardized_affilname"] = df_aff["affilname_es"].map(name_map)
df_aff[df_aff["afid"].isin(similar_epn)]

,afid,affilname,affiliation-city,affiliation-country,affilname_es,affil_clean,fingerprint,standardized_affilname
71,60072054,Escuela Politécnica Nacional,Quito,Ecuador,Escuela Politécnica Nacional,escuela politecnica nacional,escuela nacional politecnica,Escuela Superior Politécnica Agropecuaria de M...
1743,131445344,Escuela Politécnic a Nacional,Quito,Ecuador,Escuela Politécnic a Nacional,escuela politecnic a nacional,a escuela nacional politecnic,Escuela Superior Politécnica Agropecuaria de M...
1766,115317659,Escuela Poliécnica Nacional,Quito,Ecuador,Escuela Poliécnica Nacional,escuela poliecnica nacional,escuela nacional poliecnica,Escuela Poliécnica Nacional
2308,132291228,Escuela Politécnica Nacional (National Polytec...,NaN,Ecuador,Escuela Politécnica Nacional (National Polytec...,escuela politecnica nacional national polytech...,epn escuela nacional national politecnica poly...,Escuela Politécnica Nacional (National Polytec...
2340,123346277,Escuela Poltécnica Nacional,Quito,Ecuador,Escuela Poltécnica Nacional,escuela poltecnica nacional,escuela nacional poltecnica,Escuela Poltécnica Nacional
3119,130472131,Politechnical National School,Toledo,Ecuador,Politechnical National School,politechnical national school,national politechnical school,Politechnical National School
3190,121697195,Escuela Politécnica,NaN,Ecuador,Escuela Politécnica,escuela politecnica,escuela politecnica,Escuela Superior Politécnica Agropecuaria de M...
3547,128310067,Escuela PolitCrossed D Sign©cnica Nacional,Quito,Ecuador,Escuela PolitCrossed D Sign©cnica Nacional,escuela politcrossed d signcnica nacional,d escuela nacional politcrossed signcnica,Escuela PolitCrossed D Sign©cnica Nacional
3590,127982025,Escuela Politcnica,NaN,Ecuador,Escuela Politcnica,escuela politcnica,escuela politcnica,Escuela Politcnica
3993,131334134,Escuela Politéctnica Nacional,Quito,Ecuador,Escuela Politéctnica Nacional,escuela politectnica nacional,escuela nacional politectnica,Escuela Politéctnica Nacional


## Revisión manual antes y después de la desambiguación

Después de `Fingerprint` y `TF-IDF`, la desambiguación final se trata como una revisión manual asistida por similitud. Primero se identifican las cinco instituciones con mayor número de publicaciones en `articulos_ecuador.csv`; luego se listan manualmente los `afid` de variantes candidatas encontradas en `afiliaciones_ecuador.csv`, igual que en el ejemplo de EPN.

La revisión se realiza en dos momentos: antes de aplicar el mapeo canónico, para inspeccionar las variantes originales, y después de aplicarlo, para comprobar que cada grupo queda asociado a una afiliación canónica.

In [71]:
def parse_affiliation_ids(value):
    if pd.isna(value):
        return []

    try:
        parsed = literal_eval(value)
    except (ValueError, SyntaxError):
        return []

    if not isinstance(parsed, (list, tuple, set)):
        return []

    return [str(afid) for afid in parsed if pd.notna(afid)]


articles_path = DATA_DIR / "articulos_ecuador.csv"
if not articles_path.exists():
    articles_path = Path("articulos_ecuador.csv")

df_articles = pd.read_csv(articles_path)
affiliation_ids = (
    df_articles["affiliations"]
    .dropna()
    .apply(parse_affiliation_ids)
    .explode()
    .dropna()
)

affiliation_publication_counts = (
    affiliation_ids
    .value_counts()
    .rename_axis("afid")
    .reset_index(name="publication_count")
)

df_aff_counts = df_aff.copy()
df_aff_counts["afid"] = df_aff_counts["afid"].astype(str)

top_affiliations = affiliation_publication_counts.merge(
    df_aff_counts,
    on="afid",
    how="left",
)

top_affiliations.head(5)[
    ["afid", "affilname_es", "publication_count", "affiliation-city"]
]

,afid,affilname_es,publication_count,affiliation-city
0,60072061,Escuela Superior Politecnica del Litoral Ecuador,4917,Guayaquil
1,60072059,Universidad San Francisco de Quito,4783,Quito
2,60072054,Escuela Politécnica Nacional,4256,Quito
3,60072063,Pontificia Universidad Católica del Ecuador,4165,Quito
4,60104598,Universidad de las Fuerzas Armadas ESPE,3659,Sangolquí


In [72]:
top_institution_similar_affiliations = {
    "Escuela Superior Politecnica del Litoral Ecuador": [
        60072061, 129771144, 128541175, 128747316,
        126135406, 127808086, 131258983,
    ],
    "Universidad San Francisco de Quito": [
        60072059, 114370973, 128621323, 132049211, 106617475,
    ],
    "Escuela Politécnica Nacional": [
        60072054, 132291228, 130838989,
        124101307, 130722865, 130722929,
    ],
    "Pontificia Universidad Católica del Ecuador": [
        60072063, 131553112, 133156531, 127377063,
        123368997, 127632151, 120180587, 117155384, 108558436,
    ],
    "Universidad de las Fuerzas Armadas ESPE": [
        60104598, 133151254, 132413089, 128956608,
        131453317, 126169807, 128529271, 115392390,
        122733307, 119050708,
    ],
}

# Mantener los ids como texto facilita el cruce con listas de Scopus leídas desde CSV.
top_institution_similar_affiliations = {
    institution: [str(afid) for afid in afids]
    for institution, afids in top_institution_similar_affiliations.items()
}

In [73]:
top_rank_lookup = (
    top_affiliations.head(5)
    .assign(publication_rank=lambda df: range(1, len(df) + 1))
    .set_index("affilname_es")[["publication_rank", "publication_count"]]
    .to_dict(orient="index")
)

manual_review_before_frames = []

for institution, afids in top_institution_similar_affiliations.items():
    print(f"Antes de desambiguar: {institution} ({len(afids)} afid similares)")

    similar_rows = (
        df_aff_counts[df_aff_counts["afid"].isin(afids)]
        .copy()
        .assign(
            review_stage="before_disambiguation",
            institution=institution,
            publication_rank=top_rank_lookup[institution]["publication_rank"],
            publication_count=top_rank_lookup[institution]["publication_count"],
        )
        .loc[:, [
            "review_stage",
            "publication_rank",
            "publication_count",
            "institution",
            "afid",
            "affilname_es",
            "affiliation-city",
            "affiliation-country",
        ]]
        .sort_values(["publication_rank", "affilname_es", "afid"])
    )

    display(similar_rows)
    manual_review_before_frames.append(similar_rows)

manual_review_before_disambiguation = pd.concat(
    manual_review_before_frames,
    ignore_index=True,
)

manual_review_before_disambiguation.to_csv(
    OUTPUT_DIR / "top_institution_similarity_review_before.csv",
    index=False,
)

manual_review_before_disambiguation

Antes de desambiguar: Escuela Superior Politecnica del Litoral Ecuador (7 afid similares)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,affiliation-city,affiliation-country
6671,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,129771144,ESPOL Guayaquil,Guayaquil,Ecuador
4048,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128747316,Escuela Politécnica del Litoral,NaN,Ecuador
4741,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,131258983,Escuela Superior Polit cnica Del Litoral,Guayaquil,Ecuador
4639,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,127808086,Escuela Superior Polit cnica Del Litoral ESPOL,NaN,Ecuador
12,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,Guayaquil,Ecuador
2911,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,126135406,Escuela Superior Politécnicadel Litoral,NaN,Ecuador
4864,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128541175,Espol University,NaN,Ecuador


Antes de desambiguar: Universidad San Francisco de Quito (5 afid similares)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,affiliation-city,affiliation-country
1464,before_disambiguation,2,4783,Universidad San Francisco de Quito,132049211,San Francisco de Quito University,Cumbayá,Ecuador
945,before_disambiguation,2,4783,Universidad San Francisco de Quito,128621323,USFQ,NaN,Ecuador
471,before_disambiguation,2,4783,Universidad San Francisco de Quito,114370973,Universidad San Francisco de Quinto,Quito,Ecuador
13,before_disambiguation,2,4783,Universidad San Francisco de Quito,60072059,Universidad San Francisco de Quito,Quito,Ecuador
1607,before_disambiguation,2,4783,Universidad San Francisco de Quito,106617475,University of San Francisco de Quito,Quito,Ecuador


Antes de desambiguar: Escuela Politécnica Nacional (6 afid similares)


...

Antes de desambiguar: Pontificia Universidad Católica del Ecuador (9 afid similares)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,affiliation-city,affiliation-country
1306,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,133156531,Catholic University of Ecuador (PUCE),NaN,Ecuador
6763,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,117155384,Pontificia Univ Católica Ecuador,NaN,Ecuador
6638,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,120180587,Pontificia Universid Catolica del Ecuador,NaN,Ecuador
7122,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,108558436,Pontificia Universidad Cató lica del Ecuador,Quito,Ecuador
3903,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,127632151,Pontificia Universidad Católica del Ecuado,NaN,Ecuador
47,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,60072063,Pontificia Universidad Católica del Ecuador,Quito,Ecuador
736,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,131553112,Potinficia Universidad Católica del Ecuador,NaN,Ecuador
2031,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,127377063,Universidad Católica del Ecuador,Sede Quito,Ecuador
3378,before_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,123368997,Universidad Católica del Ecuador (PUCE),NaN,Ecuador


Antes de desambiguar: Universidad de las Fuerzas Armadas ESPE (10 afid similares)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,affiliation-city,affiliation-country
3953,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,122733307,ESPE,Sangolqui,Ecuador
1893,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,126169807,ESPE,Sangolquí,Ecuador
1708,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,131453317,ESPE University,NaN,Ecuador
56,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,133151254,Polytechnic School of the Army ESPE,Sangolquí,Ecuador
6485,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,119050708,Universidad de Fuerzas Amadas ESPE,Quito,Ecuador
42,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,60104598,Universidad de las Fuerzas Armadas ESPE,Sangolquí,Ecuador
1106,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,128956608,Universidad de las Fuerzas Armadas del Ecuador...,Sangolquí,Ecuador
2795,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,128529271,University of Armed Forces – ESPE,NaN,Ecuador
1055,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,132413089,University of the Armed Forced ESPE,Salgolquí,Ecuador
2572,before_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,115392390,University of the Armed Forces,Sangolqui,Ecuador


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,affiliation-city,affiliation-country
0,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,129771144,ESPOL Guayaquil,Guayaquil,Ecuador
1,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128747316,Escuela Politécnica del Litoral,NaN,Ecuador
2,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,131258983,Escuela Superior Polit cnica Del Litoral,Guayaquil,Ecuador
3,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,127808086,Escuela Superior Polit cnica Del Litoral ESPOL,NaN,Ecuador
4,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,Guayaquil,Ecuador
5,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,126135406,Escuela Superior Politécnicadel Litoral,NaN,Ecuador
6,before_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128541175,Espol University,NaN,Ecuador
7,before_disambiguation,2,4783,Universidad San Francisco de Quito,132049211,San Francisco de Quito University,Cumbayá,Ecuador
8,before_disambiguation,2,4783,Universidad San Francisco de Quito,128621323,USFQ,NaN,Ecuador
9,before_disambiguation,2,4783,Universidad San Francisco de Quito,114370973,Universidad San Francisco de Quinto,Quito,Ecuador


In [74]:
affiliation_lookup = (
    df_aff_counts
    .drop_duplicates(subset="afid")
    .set_index("afid")
)

canonical_afid_by_institution = {
    institution: afids[0]
    for institution, afids in top_institution_similar_affiliations.items()
}

manual_canonical_affiliation_map = {}
manual_group_by_afid = {}

for institution, afids in top_institution_similar_affiliations.items():
    canonical_afid = canonical_afid_by_institution[institution]
    canonical_name = affiliation_lookup.at[canonical_afid, "affilname_es"]

    for afid in afids:
        manual_canonical_affiliation_map[afid] = {
            "institution": institution,
            "canonical_afid_manual": canonical_afid,
            "canonical_affilname_manual": canonical_name,
        }
        manual_group_by_afid[afid] = institution

def manual_canonical_value(afid, field, default_value):
    return manual_canonical_affiliation_map.get(str(afid), {}).get(field, default_value)


df_aff_disambiguated = df_aff_counts.copy()
df_aff_disambiguated["canonical_afid_manual"] = df_aff_disambiguated["afid"].apply(
    lambda afid: manual_canonical_value(afid, "canonical_afid_manual", str(afid))
)
df_aff_disambiguated["canonical_affilname_manual"] = df_aff_disambiguated.apply(
    lambda row: manual_canonical_value(row["afid"], "canonical_affilname_manual", row["affilname_es"]),
    axis=1,
)
df_aff_disambiguated["manual_disambiguation_group"] = df_aff_disambiguated["afid"].map(
    manual_group_by_afid
)
df_aff_disambiguated["changed_by_manual_disambiguation"] = (
    df_aff_disambiguated["afid"] != df_aff_disambiguated["canonical_afid_manual"]
)

manual_canonical_affiliation_map_df = pd.DataFrame.from_dict(
    manual_canonical_affiliation_map,
    orient="index",
).rename_axis("afid").reset_index()

manual_canonical_affiliation_map_df.to_csv(
    OUTPUT_DIR / "manual_canonical_affiliation_map.csv",
    index=False,
)

manual_canonical_affiliation_map_df.head(10)

,afid,institution,canonical_afid_manual,canonical_affilname_manual
0,60072061,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador
1,129771144,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador
2,128541175,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador
3,128747316,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador
4,126135406,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador
5,127808086,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador
6,131258983,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador
7,60072059,Universidad San Francisco de Quito,60072059,Universidad San Francisco de Quito
8,114370973,Universidad San Francisco de Quito,60072059,Universidad San Francisco de Quito
9,128621323,Universidad San Francisco de Quito,60072059,Universidad San Francisco de Quito


In [75]:
manual_review_after_frames = []

for institution, afids in top_institution_similar_affiliations.items():
    print(f"Después de desambiguar: {institution} ({len(afids)} afid similares)")

    similar_rows = (
        df_aff_disambiguated[df_aff_disambiguated["afid"].isin(afids)]
        .copy()
        .assign(
            review_stage="after_disambiguation",
            institution=institution,
            publication_rank=top_rank_lookup[institution]["publication_rank"],
            publication_count=top_rank_lookup[institution]["publication_count"],
        )
        .loc[:, [
            "review_stage",
            "publication_rank",
            "publication_count",
            "institution",
            "afid",
            "affilname_es",
            "canonical_afid_manual",
            "canonical_affilname_manual",
            "changed_by_manual_disambiguation",
            "affiliation-city",
            "affiliation-country",
        ]]
        .sort_values(["publication_rank", "canonical_affilname_manual", "affilname_es", "afid"])
    )

    display(similar_rows)
    manual_review_after_frames.append(similar_rows)

manual_review_after_disambiguation = pd.concat(
    manual_review_after_frames,
    ignore_index=True,
)

top_institution_similarity_review = manual_review_after_disambiguation.copy()

manual_review_after_disambiguation.to_csv(
    OUTPUT_DIR / "top_institution_similarity_review_after.csv",
    index=False,
)
top_institution_similarity_review.to_csv(
    OUTPUT_DIR / "top_institution_similarity_review.csv",
    index=False,
)

manual_review_after_disambiguation

Después de desambiguar: Escuela Superior Politecnica del Litoral Ecuador (7 afid similares)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,canonical_afid_manual,canonical_affilname_manual,changed_by_manual_disambiguation,affiliation-city,affiliation-country
6671,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,129771144,ESPOL Guayaquil,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,Guayaquil,Ecuador
4048,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128747316,Escuela Politécnica del Litoral,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,NaN,Ecuador
4741,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,131258983,Escuela Superior Polit cnica Del Litoral,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,Guayaquil,Ecuador
4639,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,127808086,Escuela Superior Polit cnica Del Litoral ESPOL,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,NaN,Ecuador
12,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Guayaquil,Ecuador
2911,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,126135406,Escuela Superior Politécnicadel Litoral,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,NaN,Ecuador
4864,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128541175,Espol University,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,NaN,Ecuador


Después de desambiguar: Universidad San Francisco de Quito (5 afid similares)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,canonical_afid_manual,canonical_affilname_manual,changed_by_manual_disambiguation,affiliation-city,affiliation-country
1464,after_disambiguation,2,4783,Universidad San Francisco de Quito,132049211,San Francisco de Quito University,60072059,Universidad San Francisco de Quito,True,Cumbayá,Ecuador
945,after_disambiguation,2,4783,Universidad San Francisco de Quito,128621323,USFQ,60072059,Universidad San Francisco de Quito,True,NaN,Ecuador
471,after_disambiguation,2,4783,Universidad San Francisco de Quito,114370973,Universidad San Francisco de Quinto,60072059,Universidad San Francisco de Quito,True,Quito,Ecuador
13,after_disambiguation,2,4783,Universidad San Francisco de Quito,60072059,Universidad San Francisco de Quito,60072059,Universidad San Francisco de Quito,False,Quito,Ecuador
1607,after_disambiguation,2,4783,Universidad San Francisco de Quito,106617475,University of San Francisco de Quito,60072059,Universidad San Francisco de Quito,True,Quito,Ecuador


Después de desambiguar: Escuela Politécnica Nacional (6 afid similares)


...

Después de desambiguar: Pontificia Universidad Católica del Ecuador (9 afid similares)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,canonical_afid_manual,canonical_affilname_manual,changed_by_manual_disambiguation,affiliation-city,affiliation-country
1306,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,133156531,Catholic University of Ecuador (PUCE),60072063,Pontificia Universidad Católica del Ecuador,True,NaN,Ecuador
6763,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,117155384,Pontificia Univ Católica Ecuador,60072063,Pontificia Universidad Católica del Ecuador,True,NaN,Ecuador
6638,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,120180587,Pontificia Universid Catolica del Ecuador,60072063,Pontificia Universidad Católica del Ecuador,True,NaN,Ecuador
7122,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,108558436,Pontificia Universidad Cató lica del Ecuador,60072063,Pontificia Universidad Católica del Ecuador,True,Quito,Ecuador
3903,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,127632151,Pontificia Universidad Católica del Ecuado,60072063,Pontificia Universidad Católica del Ecuador,True,NaN,Ecuador
47,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,60072063,Pontificia Universidad Católica del Ecuador,60072063,Pontificia Universidad Católica del Ecuador,False,Quito,Ecuador
736,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,131553112,Potinficia Universidad Católica del Ecuador,60072063,Pontificia Universidad Católica del Ecuador,True,NaN,Ecuador
2031,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,127377063,Universidad Católica del Ecuador,60072063,Pontificia Universidad Católica del Ecuador,True,Sede Quito,Ecuador
3378,after_disambiguation,4,4165,Pontificia Universidad Católica del Ecuador,123368997,Universidad Católica del Ecuador (PUCE),60072063,Pontificia Universidad Católica del Ecuador,True,NaN,Ecuador


Después de desambiguar: Universidad de las Fuerzas Armadas ESPE (10 afid similares)


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,canonical_afid_manual,canonical_affilname_manual,changed_by_manual_disambiguation,affiliation-city,affiliation-country
3953,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,122733307,ESPE,60104598,Universidad de las Fuerzas Armadas ESPE,True,Sangolqui,Ecuador
1893,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,126169807,ESPE,60104598,Universidad de las Fuerzas Armadas ESPE,True,Sangolquí,Ecuador
1708,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,131453317,ESPE University,60104598,Universidad de las Fuerzas Armadas ESPE,True,NaN,Ecuador
56,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,133151254,Polytechnic School of the Army ESPE,60104598,Universidad de las Fuerzas Armadas ESPE,True,Sangolquí,Ecuador
6485,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,119050708,Universidad de Fuerzas Amadas ESPE,60104598,Universidad de las Fuerzas Armadas ESPE,True,Quito,Ecuador
42,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,60104598,Universidad de las Fuerzas Armadas ESPE,60104598,Universidad de las Fuerzas Armadas ESPE,False,Sangolquí,Ecuador
1106,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,128956608,Universidad de las Fuerzas Armadas del Ecuador...,60104598,Universidad de las Fuerzas Armadas ESPE,True,Sangolquí,Ecuador
2795,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,128529271,University of Armed Forces – ESPE,60104598,Universidad de las Fuerzas Armadas ESPE,True,NaN,Ecuador
1055,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,132413089,University of the Armed Forced ESPE,60104598,Universidad de las Fuerzas Armadas ESPE,True,Salgolquí,Ecuador
2572,after_disambiguation,5,3659,Universidad de las Fuerzas Armadas ESPE,115392390,University of the Armed Forces,60104598,Universidad de las Fuerzas Armadas ESPE,True,Sangolqui,Ecuador


,review_stage,publication_rank,publication_count,institution,afid,affilname_es,canonical_afid_manual,canonical_affilname_manual,changed_by_manual_disambiguation,affiliation-city,affiliation-country
0,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,129771144,ESPOL Guayaquil,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,Guayaquil,Ecuador
1,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128747316,Escuela Politécnica del Litoral,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,NaN,Ecuador
2,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,131258983,Escuela Superior Polit cnica Del Litoral,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,Guayaquil,Ecuador
3,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,127808086,Escuela Superior Polit cnica Del Litoral ESPOL,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,NaN,Ecuador
4,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,60072061,Escuela Superior Politecnica del Litoral Ecuador,False,Guayaquil,Ecuador
5,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,126135406,Escuela Superior Politécnicadel Litoral,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,NaN,Ecuador
6,after_disambiguation,1,4917,Escuela Superior Politecnica del Litoral Ecuador,128541175,Espol University,60072061,Escuela Superior Politecnica del Litoral Ecuador,True,NaN,Ecuador
7,after_disambiguation,2,4783,Universidad San Francisco de Quito,132049211,San Francisco de Quito University,60072059,Universidad San Francisco de Quito,True,Cumbayá,Ecuador
8,after_disambiguation,2,4783,Universidad San Francisco de Quito,128621323,USFQ,60072059,Universidad San Francisco de Quito,True,NaN,Ecuador
9,after_disambiguation,2,4783,Universidad San Francisco de Quito,114370973,Universidad San Francisco de Quinto,60072059,Universidad San Francisco de Quito,True,Quito,Ecuador


## Representative cases for affiliation disambiguation

The following table summarizes one manually adjudicated pair for each of the five Ecuadorian educational institutions with the highest publication counts. The format follows the paper table structure: two observed affiliation strings, the disambiguation decision, the ground-truth label, and the ambiguity source.

In [76]:
paper_case_columns = [
    "Affiliation A",
    "Affiliation B",
    "Decision",
    "Ground Truth",
    "Explanation",
]

affiliation_lookup = (
    df_aff_disambiguated
    .drop_duplicates(subset="afid")
    .set_index("afid")
)


def affiliation_label(afid):
    afid = str(afid)
    if afid not in affiliation_lookup.index:
        raise KeyError(f"Affiliation id {afid} was not found in the affiliations table.")
    return affiliation_lookup.at[afid, "affilname_es"]


paper_case_specs = [
    # ESPOL: highest publication count in this dataset.
    ("60072061", "129771144", "Merge", "Same", "Abbreviation / city qualifier"),
    # USFQ: one-character typo in the institution name.
    ("60072059", "114370973", "Merge", "Same", "Typographical error"),
    # EPN: Spanish and English variants with acronym.
    ("60072054", "124101307", "Merge", "Same", "Translation / acronym"),
    # PUCE: typo and acronym-bearing variant.
    ("60072063", "127632151", "Merge", "Same", "Typographical error"),
    # ESPE: Spanish name and English translation.
    ("60104598", "133151254", "Merge", "Same", "Translation / acronym"),
]

paper_affiliation_cases = pd.DataFrame(
    [
        {
            "Affiliation A": affiliation_label(afid_a),
            "Affiliation B": affiliation_label(afid_b),
            "Decision": decision,
            "Ground Truth": ground_truth,
            "Explanation": explanation,
        }
        for afid_a, afid_b, decision, ground_truth, explanation in paper_case_specs
    ],
    columns=paper_case_columns,
)

display(paper_affiliation_cases)

paper_affiliation_cases.to_csv(
    OUTPUT_DIR / "affiliation_disambiguation_cases.csv",
    index=False,
)


def latex_escape(value):
    escape_map = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(escape_map.get(char, char) for char in str(value))


def dataframe_to_latex(df, caption, label):
    column_spec = "l" * len(df.columns)
    latex_lines = [
        r"\begin{table}",
        r"\centering",
        rf"\caption{{{latex_escape(caption)}}}",
        rf"\label{{{latex_escape(label)}}}",
        rf"\begin{{tabular}}{{{column_spec}}}",
        r"\hline",
        " & ".join(latex_escape(column) for column in df.columns) + r" \\",
        r"\hline",
    ]

    for _, row in df.iterrows():
        latex_lines.append(
            " & ".join(latex_escape(row[column]) for column in df.columns) + r" \\",
        )

    latex_lines.extend([
        r"\hline",
        r"\end{tabular}",
        r"\end{table}",
    ])
    return "\n".join(latex_lines) + "\n"


latex_table = dataframe_to_latex(
    paper_affiliation_cases,
    caption="Representative cases for affiliation disambiguation.",
    label="tab:affiliation-disambiguation-cases",
)
(OUTPUT_DIR / "affiliation_disambiguation_cases.tex").write_text(
    latex_table,
    encoding="utf-8",
)

,Affiliation A,Affiliation B,Decision,Ground Truth,Explanation
0,Escuela Superior Politecnica del Litoral Ecuador,ESPOL Guayaquil,Merge,Same,Abbreviation / city qualifier
1,Universidad San Francisco de Quito,Universidad San Francisco de Quinto,Merge,Same,Typographical error
2,Escuela Politécnica Nacional,National Polytechnic School (EPN),Merge,Same,Translation / acronym
3,Pontificia Universidad Católica del Ecuador,Pontificia Universidad Católica del Ecuado,Merge,Same,Typographical error
4,Universidad de las Fuerzas Armadas ESPE,Polytechnic School of the Army ESPE,Merge,Same,Translation / acronym


862